<p style="font-size:16pt; font-weight:bold; color:red; padding-bottom:20px; float:right">Please rename this file before editing!</p>

# Session 10 — Chance on Purpose

So far every program in this course gave the same answer each time it ran.
This session is about programs that use chance on purpose: a **simulation**
answers a question by running an experiment many times and counting. It makes
problems approachable before the mathematics is.

Six ideas, one part each:

| Part | Idea | In one sentence |
|---|---|---|
| 1 | Simulate to answer | Repeat an experiment many times; the share of successes estimates the probability. |
| 2 | Pseudorandom numbers | The "random" numbers follow from a seed, so a run can be repeated exactly. |
| 3 | One trial, many trials | Write the experiment once, as a function of a generator; run it as often as needed. |
| 4 | An estimate is uncertain | It changes from run to run, and its standard error says by how much. |
| 5 | More trials, slowly better | The error shrinks with the square root of the number of trials. |
| 6 | Fair comparisons and resampling | Compare options on the same random draws; apply the same idea to real data. |

**How this session's notebooks fit together**

| Notebook | What it is |
|---|---|
| `01_Chance_on_Purpose` (this one) | The ideas, on problems whose answers are known |
| `02_Help_Desk_Example` | One simulation study from model to recommendation |
| `03_Simulation_Reference` | A lookup guide to keep open while you do Homework 9 |

**What is not repeated here.** Mean, standard deviation, quantiles, and every
plot used below are Session 9
(`../09-Visualization-Descriptive-Statistics/`). Arrays and whole-column
arithmetic are Sessions 7 and 8.

**Before you start**

- Set the kernel (upper right corner) to **Conda Python 3.12**.
- Run the cells in order with **Shift + Enter**. Before each cell, guess what
  it will show.

In [ ]:
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.options.display.float_format = "{:,.4f}".format

---

## Part 1 — Simulate to answer

Roll two dice. How likely is it that they add up to 7?

You may know the answer: 6 of the 36 equally likely pairs add up to 7, so 1/6,
about 0.1667. Good. A problem with a known answer is the right place to learn
a method, because you can see whether the method works.

The method: roll the two dice many times, and count.

In [ ]:
rolls = 10_000
sevens = 0
for roll in range(rolls):
    first = random.randint(1, 6)          # a whole number from 1 to 6, both included
    second = random.randint(1, 6)
    if first + second == 7:
        sevens += 1

print(sevens, "sevens in", rolls, "rolls:", sevens / rolls)

Close to 0.1667, and not equal to it. **Run the cell again.** A different
number, also close.

That is the whole idea, and both halves of it matter:

- the share of successes in many trials **estimates** the probability;
- the estimate is **not the answer**: it moves every time.

A method built on this is called a **Monte Carlo** method, after the casino.
The rest of the session is about making the first half convenient and the
second half honest.

**✏️ Try it:** Change the question to "at least one six" and run it. The exact answer is 11/36. How close do you get with 100 rolls? With 100,000?

---

## Part 2 — Pseudorandom numbers

A computer follows instructions exactly, so where does chance come from? It
does not. `random.randint` is computed by a recipe that turns one number into
the next, in a sequence so scrambled that it passes for random. The first
number of the sequence is the **seed**.

In [ ]:
random.seed(2026)
print([random.randint(1, 6) for roll in range(10)])

random.seed(2026)
print([random.randint(1, 6) for roll in range(10)])

random.seed(2027)
print([random.randint(1, 6) for roll in range(10)])

The same seed gives the same "random" numbers, every time, on every machine.
Another seed gives others. With no seed, Python takes one from the clock, and
each run differs.

That is why they are called **pseudorandom**, and it is a feature. A result
that cannot be repeated cannot be checked, debugged or graded. Fix the seed
and a simulation becomes as repeatable as any other program.

### A generator you can hold

`random.seed(...)` sets one generator shared by the entire program. If any
other code draws a number in between, yours changes. The better tool is a
generator that is a **value**: you make it, keep it in a variable, and hand it
to whatever needs random numbers. numpy's is the one this course uses.

In [ ]:
rng = np.random.default_rng(2026)         # a generator, started from seed 2026

print(rng.integers(1, 7, size=10))        # ten whole numbers from 1 to 6
print(rng.integers(1, 7, size=10))        # ten more: it remembers where it is

Three things to notice.

- `rng.integers(1, 7)` gives 1 to 6. The upper end is **left out**, like
  `range(1, 7)`. (`random.randint(1, 6)` includes it. The two libraries
  disagree, and it is a classic source of dice that never show a six.)
- `size=10` asks for ten numbers at once, as an array.
- The second line continued the sequence. A generator has a **state**: it
  knows how far along it is.

In [ ]:
first = np.random.default_rng(7)
second = np.random.default_rng(7)

print(first.integers(1, 7, size=5), first.integers(1, 7, size=5))
print(second.integers(1, 7, size=10))

Same seed, same ten numbers, whether they are drawn five at a time or ten at
once.

### The toolbox

| You want | Write this |
|---|---|
| a whole number from `low` to `high − 1` | `rng.integers(low, high)` |
| a number between 0 and 1 | `rng.random()` |
| one of a list of things | `rng.choice(things)` |
| several, the same one possibly twice | `rng.choice(things, size=3)` |
| several different ones | `rng.choice(things, size=3, replace=False)` |
| the things in a random order | `rng.permutation(things)` |
| a number that clusters around a mean | `rng.normal(mean, spread)` |
| a count of arrivals with a given mean | `rng.poisson(mean)` |

Every one of them takes `size=` for many at once. The reference notebook has
the full list.

In [ ]:
names = ["Ana", "Ben", "Chi", "Dev", "Eli"]

print("one:                 ", rng.choice(names))
print("three, with repeats: ", rng.choice(names, size=3))
print("three different:     ", rng.choice(names, size=3, replace=False))
print("a random order:      ", rng.permutation(names))

**With replacement** means a thing can be picked again, like rolling a die.
**Without replacement** means it cannot, like dealing cards. Which one the
problem needs is the first question to ask of any sampling.

**✏️ Try it:** Deal a hand of 5 cards from a deck of 52 (use the numbers 0 to 51). With or without replacement? Then roll 5 dice. With or without?

---

## Part 3 — One trial, many trials

A simulation has the same two layers every time:

1. **one trial**: a function that runs the experiment once and returns its
   outcome. It takes the generator as an argument.
2. **many trials**: a loop that calls it and keeps the outcomes.

The Monty Hall problem makes a good trial. A game show has three doors, a car
behind one. You pick a door. The host, who knows where the car is, opens one
of the *other* doors, always one with no car. Then he offers: stay, or switch
to the remaining door?

Most people say it makes no difference. Do not argue. Simulate.

In [ ]:
def monty_hall(rng, switch):
    """Play one game.  Return True when the player wins the car."""
    car = rng.integers(0, 3)
    pick = rng.integers(0, 3)

    # The host opens a door that is neither the pick nor the car.
    closed_to_host = {int(car), int(pick)}
    opened = rng.choice([door for door in range(3) if door not in closed_to_host])

    if switch:
        pick = [door for door in range(3) if door not in (int(pick), int(opened))][0]
    return pick == car


rng = np.random.default_rng(1)
print([bool(monty_hall(rng, switch=True)) for game in range(10)])

Ten games, ten outcomes. One trial says almost nothing. Now the second layer,
written once and usable for any trial:

In [ ]:
def run_trials(trial, trials, rng):
    """Call trial(rng) `trials` times; return the outcomes as floats."""
    outcomes = np.empty(trials)
    for number in range(trials):
        outcomes[number] = trial(rng)
    return outcomes


rng = np.random.default_rng(1)
switching = run_trials(lambda rng: monty_hall(rng, switch=True), 10_000, rng)
staying = run_trials(lambda rng: monty_hall(rng, switch=False), 10_000, rng)

print("win by switching:", switching.mean())
print("win by staying:  ", staying.mean())

True is stored as 1.0 and False as 0.0, so **the mean of the outcomes is the
share of wins**. Switching wins two times in three. (Your first pick is right
one time in three; when it is wrong, switching always wins.)

Three choices in that code are worth copying.

- **The trial takes `rng`.** It does not make its own generator and does not
  use a global one. Whoever calls it decides where the randomness comes from.
- **`run_trials` knows nothing about doors.** `trial` is a function passed as
  an argument; the same loop runs dice, doors or a coffee cart.
- **One generator, made once, at the top.** Everything follows from the one
  seed.

Here is what happens when the second rule is broken:

In [ ]:
def broken_trial(rng_is_ignored):
    rng = np.random.default_rng(1)            # a fresh generator on every call
    return monty_hall(rng, switch=True)


outcomes = run_trials(broken_trial, 1000, np.random.default_rng(5))
print("share of wins:", outcomes.mean(), "  different outcomes:", np.unique(outcomes))

A thousand copies of one game. The program runs, prints a number, and the
number is 0 or 1. A generator made inside the trial restarts the sequence on
every call.

**✏️ Try it:** Write `two_dice(rng)` that returns True when two dice add up to 7, and run it 10,000 times with `run_trials`.

---

## Part 4 — An estimate is uncertain

Run the dice simulation with twenty different seeds, 1,000 rolls each.

In [ ]:
def seven(rng):
    return rng.integers(1, 7, size=2).sum() == 7


estimates = [run_trials(seven, 1000, np.random.default_rng(seed)).mean()
             for seed in range(20)]

print(np.round(estimates, 3))
print("smallest", min(estimates), " largest", max(estimates), " exact", round(1 / 6, 4))

Twenty honest runs of a correct program, and the answers range over several
hundredths. If you had run it once and reported three decimals, two of them
would have been noise.

How much an estimate moves from run to run is its **standard error**, and you
do not need twenty runs to know it. One run is enough:

```text
standard error = standard deviation of the outcomes / sqrt(number of trials)
```

The standard deviation says how much *one trial* varies. Averaging many trials
cancels most of that, and the square root says how much is left.

In [ ]:
outcomes = run_trials(seven, 1000, np.random.default_rng(0))

mean = outcomes.mean()
se = outcomes.std(ddof=1) / np.sqrt(len(outcomes))

print("estimate        ", mean)
print("standard error  ", round(se, 4), "   <- computed from this one run")
print("spread of the 20 estimates above:", round(np.std(estimates, ddof=1), 4))

The standard error from one run matches the spread of the twenty. That is what
it is for.

(`ddof=1` asks numpy for the sample standard deviation, dividing by count − 1.
Session 9: numpy divides by count unless told otherwise.)

An **interval** makes it a statement. Go about two standard errors to each
side of the estimate:

In [ ]:
low, high = mean - 1.96 * se, mean + 1.96 * se
print("{:.3f}, interval {:.3f} to {:.3f}, from {:,} trials".format(mean, low, high, len(outcomes)))

What does the interval claim? Not that the truth is certainly inside. The
claim is about the method: if the whole simulation were repeated with many
seeds, about 95 of 100 such intervals would contain the true value. Check it,
since here the truth is known:

In [ ]:
inside = 0
runs = 200
for seed in range(runs):
    outcomes = run_trials(seven, 1000, np.random.default_rng(seed))
    mean = outcomes.mean()
    se = outcomes.std(ddof=1) / np.sqrt(len(outcomes))
    if mean - 1.96 * se <= 1 / 6 <= mean + 1.96 * se:
        inside += 1

print(inside, "of", runs, "intervals contain 1/6")

About 95 in 100, as promised, which also means about one in twenty misses. A
simulated number with its interval is an honest report. Without the interval,
nobody can tell 0.167 ± 0.002 from 0.167 ± 0.05.

**✏️ Try it:** How many of the 200 intervals contain the truth if you use 1.0 in place of 1.96? And with 3.0?

---

## Part 5 — More trials, slowly better

Watch one long run settle. The running share after each roll:

In [ ]:
outcomes = run_trials(seven, 20_000, np.random.default_rng(3))
trials = np.arange(1, len(outcomes) + 1)
running = np.cumsum(outcomes) / trials

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.plot(trials, running, color="steelblue", label="share of sevens so far")
ax.axhline(1 / 6, color="firebrick", linestyle="--", label="exact, 1/6")
ax.set_xscale("log")
ax.set_ylim(0, 0.4)
ax.set_xlabel("rolls so far (logarithmic scale)")
ax.set_ylabel("share of sevens")
ax.set_title("The estimate settles, and never quite stops moving")
ax.legend()
fig.tight_layout()

And the price of precision, as a table:

In [ ]:
rng = np.random.default_rng(4)
rows = []
for count in (100, 1_000, 10_000, 100_000):
    outcomes = run_trials(seven, count, rng)
    se = outcomes.std(ddof=1) / np.sqrt(count)
    rows.append({"trials": count, "estimate": outcomes.mean(), "se": se,
                 "low": outcomes.mean() - 1.96 * se, "high": outcomes.mean() + 1.96 * se})
pd.DataFrame(rows)

Each row has ten times the trials of the one above, and a standard error about
three times smaller: the square root of ten. **A hundred times the work buys
one more decimal place.**

| To make the error | you need |
|---|---|
| half as large | 4 times the trials |
| a tenth as large | 100 times the trials |
| a hundredth as large | 10,000 times the trials |

So "how many trials?" has an answer: enough that the interval is narrower than
the difference you care about, and no more. And more decimals in the printout
are not more truth. Print as many digits as the standard error supports.

### All the trials at once

A Python loop that calls a function 100,000 times is slow. numpy can often
draw every trial in one call and do the arithmetic on whole arrays (Session
8's vectorized thinking).

In [ ]:
rng = np.random.default_rng(5)

start = time.perf_counter()
looped = run_trials(seven, 100_000, rng)
loop_seconds = time.perf_counter() - start

start = time.perf_counter()
dice = rng.integers(1, 7, size=(100_000, 2))        # 100,000 rows of two dice
vectorized = dice.sum(axis=1) == 7
vector_seconds = time.perf_counter() - start

print("loop:       {:.3f} s   estimate {:.4f}".format(loop_seconds, looped.mean()))
print("vectorized: {:.3f} s   estimate {:.4f}".format(vector_seconds, vectorized.mean()))

Hundreds of times faster. The two estimates differ, though both used the same
generator: they drew their numbers in a different order. **Two correct
programs with the same seed can print different results.** Neither is wrong,
and both are within their standard error of 1/6. This is why a simulation is
never tested by comparing it with someone else's digits.

Write the loop version first. It is easier to get right, and it is what you
check the fast one against.

**✏️ Try it:** Vectorize "at least one six in four rolls": `rng.integers(1, 7, size=(100_000, 4))`, then `(dice == 6).any(axis=1)`. The exact answer is 1 − (5/6)⁴.

---

## Part 6 — Fair comparisons, and resampling

### Compare options on the same draws

Two bakers each claim a better way to cut a cake, and you test one on Monday's
customers and the other on Saturday's. Whatever you measure, you cannot tell
the method from the day.

Simulations have the same problem, and a cure that real experiments rarely
have: **give both options exactly the same random numbers.** A shop sells a
random number of umbrellas a day, more when it rains. Should it stock 22 or
23?

In [ ]:
def profit(stock, demand):
    """$12 per umbrella sold, $5 per umbrella stocked, for every day."""
    return 12 * np.minimum(stock, demand) - 5 * stock


rng = np.random.default_rng(6)

print("each option on its own random days:")
for attempt in range(4):
    days_a = rng.poisson(22, size=2000)
    days_b = rng.poisson(22, size=2000)
    print("   23 minus 22: {:+.2f}".format(profit(23, days_a).mean() - profit(22, days_b).mean()))

print("both options on the same days:")
for attempt in range(4):
    days = rng.poisson(22, size=2000)
    print("   23 minus 22: {:+.2f}".format((profit(23, days) - profit(22, days)).mean()))

The first four answers are scattered over several dollars and do not even
agree on the sign. The second four agree with each other to within a few
tenths.
Nothing about the shop changed: in the second version, a busy day is busy for
both options, so its luck cancels out of the difference.

This is called **common random numbers**. It is the reason to keep "draw the
random inputs" and "compute the result" in separate functions: draw once, then
try every option on the same draw.

### Resampling real data

The last idea turns the machinery around. You have real data, a sample of 12
delivery times in minutes, and its mean. How far would the mean move with
another 12?

In [ ]:
deliveries = np.array([22, 25, 19, 31, 24, 27, 58, 21, 26, 23, 29, 20])
print("mean", deliveries.mean(), " median", np.median(deliveries))

You cannot draw another 12 from the world. But the sample is your best picture
of the world, so draw from **it**: 12 values, *with replacement*, so that some
appear twice and some not at all. Each such **resample** is a version of the
data that could as well have turned up.

In [ ]:
rng = np.random.default_rng(7)

for number in range(4):
    resample = rng.choice(deliveries, size=len(deliveries), replace=True)
    print(np.sort(resample), "  mean {:.1f}".format(resample.mean()))

In [ ]:
means = np.array([rng.choice(deliveries, size=len(deliveries), replace=True).mean()
                  for number in range(5000)])
medians = np.array([np.median(rng.choice(deliveries, size=len(deliveries), replace=True))
                    for number in range(5000)])

print("mean   {:.1f}, 95 % of resampled means   between {:.1f} and {:.1f}".format(
    deliveries.mean(), *np.quantile(means, [0.025, 0.975])))
print("median {:.1f}, 95 % of resampled medians between {:.1f} and {:.1f}".format(
    np.median(deliveries), *np.quantile(medians, [0.025, 0.975])))

This is the **bootstrap**. The interval for the mean is wide and lopsided,
because of the one 58-minute delivery: resamples that draw it twice have a
high mean, resamples that miss it a low one. The median barely notices.

The bootstrap needs no formula. It works for a median, a quantile, a
correlation, anything you can compute from a sample. Its limit is the sample
itself: it can only reshuffle what was observed, and twelve values know little
about rare events.

**✏️ Try it:** Bootstrap the **largest** delivery time. What interval do you get, and why should you not trust it?

---

## Reporting a simulated result

> A simulated number without its seed and its number of trials cannot be
> checked, and without its interval cannot be judged.

| Report | Why |
|---|---|
| the **seed** | so that anyone can get exactly your numbers |
| the **number of trials** | it decides the precision |
| an **interval**, or the standard error | how far the estimate can be trusted |
| no more **digits** than the standard error supports | 0.1674 ± 0.004 is 0.167 |
| the **model**: what was assumed | the simulation answers the model, not the world |

## Exercises

### 🧩 Exercise 1 — A known answer

Three coins are tossed. Estimate the probability of exactly two heads from
10,000 trials, with a standard error and an interval. The exact answer is 3/8.
Is it inside your interval? Report the result in one sentence, with the seed.

### 🧩 Exercise 2 — The square root, seen

For the three-coin trial, compute the standard error at 100, 400, 1,600 and
6,400 trials. Each count is four times the last. What happens to the standard
error, and how many trials would you need for a standard error of 0.001?

### 🧩 Exercise 3 — Seeds

Write a function `study(seed)` that makes one generator from the seed and
returns the three-coin estimate from 2,000 trials. Show that `study(1)` twice
gives the same number and `study(2)` another. Then find the bug in this
version, and say what it would print:

```python
def study(seed):
    outcomes = []
    for number in range(2000):
        rng = np.random.default_rng(seed)
        outcomes.append(rng.integers(0, 2, size=3).sum() == 2)
    return np.mean(outcomes)
```

### 🧩 Exercise 4 — A fair comparison

The umbrella shop of Part 6 considers stocking 22, 23 or 24. Draw 5,000 days
of demand **once** and compute the mean profit of each. Then estimate "24
minus 23" from the daily differences, with a standard error. Can your data
tell the two apart?

---

## Summary

| Idea | The habit | The tool |
|---|---|---|
| Simulate to answer | Try the method on a known answer first | a loop and a counter |
| Pseudorandom numbers | Fix the seed; hold the generator in a variable | `np.random.default_rng(seed)` |
| One trial, many trials | The trial takes `rng`; one generator, made once | `trial(rng)`, `run_trials(trial, n, rng)` |
| An estimate is uncertain | Never report it without an interval | `std(ddof=1) / sqrt(n)`, ± 1.96 standard errors |
| More trials, slowly better | 100 times the work per decimal place | a table over trial counts |
| Fair comparisons | Draw the randomness once, try every option on it | daily differences |
| Resampling | Draw from the sample, with replacement | `rng.choice(values, size=len(values))`, `np.quantile` |

### Where each idea appears in Homework 9

| Idea | Homework 9 |
|---|---|
| A known answer first; one trial from a generator | Phase 1, `exact_birthday`, `shared_birthday` |
| Many trials, one generator passed along | Phase 2, `run_trials` |
| Standard error and interval | Phase 3, `standard_error`, `estimate` |
| The square root | Phase 4, `convergence` |
| Randomness in one function, arithmetic in another | Phase 5, `draw_demand`, `profits` |
| Common random numbers, daily differences | Phase 6, `compare_batches`, `difference` |
| The bootstrap; a report that follows from its seed | Phase 7, `bootstrap_interval`, `build_report` |

**Next:** `02_Help_Desk_Example` builds a simulation with no formula behind
it, and `03_Simulation_Reference` is the lookup guide.